## Learning rate schedulers
Instead of keeping the learning rate fixed, you can decay it over time — start fast, then take smaller and smaller steps as you close in on the minimum.
```python
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Reduce LR by 0.5 every 5 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

for epoch in range(20):
    train(model, train_loader, optimizer, criterion)
    scheduler.step()   # ← call after each epoch
    print(f"Epoch {epoch+1}, LR: {scheduler.get_last_lr()[0]:.5f}")
```

Another popular option — ReduceLROnPlateau — automatically reduces LR when validation loss stops improving. Very hands-off:
```python
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

for epoch in range(20):
    _, val_loss = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)   # pass the metric you're watching
```

## Batch size — the underrated one
Larger batch size → more stable gradients, but less regularization effect and more memory needed.
Smaller batch size → noisier updates, but often generalizes better and fits in less memory.
A practical rule of thumb:

Start with 32 or 64
If training is unstable → try smaller (16)
If you have a big GPU and training is slow → try larger (128, 256)

```python
# Easy to experiment with — just change one number
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
```

### The systematic approach: grid search
Instead of guessing, try a small grid of values and pick the best:
```python
import itertools

learning_rates = [0.001, 0.01, 0.1]
batch_sizes    = [32, 64]

results = []

for lr, bs in itertools.product(learning_rates, batch_sizes):
    # Fresh model + loader for each combo
    model     = CNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader    = DataLoader(train_data, batch_size=bs, shuffle=True)

    for epoch in range(3):   # quick 3-epoch check
        train(model, loader, optimizer, criterion)

    _, val_acc = evaluate(model, val_loader, criterion)
    results.append((lr, bs, val_acc))
    print(f"lr={lr}, bs={bs} → val_acc={val_acc:.3f}")

# Sort by val accuracy
results.sort(key=lambda x: x[2], reverse=True)
print(f"\nBest: lr={results[0][0]}, bs={results[0][1]}, acc={results[0][2]:.3f}")
```

---

### Practical tuning checklist

Here's the order to tune things in — fixing issues from the most impactful to least:
```
1. Learning rate        → most impactful, try first
2. Model size           → too small = underfitting, too large = overfitting
3. Batch size           → affects stability and speed
4. Regularization       → dropout rate, weight decay
5. Scheduler            → adds a small boost once other things are stable
6. Everything else      → optimizer choice, activation functions, etc.